# Cardiovascular Disease Risk Prediction - EDA

This notebook performs Phase 1 exploratory data analysis for the hackathon project using the provided public datasets:

- `data/raw/heart_processed.csv`: compact heart disease dataset aligned with Streamlit patient vitals.
- `data/raw/cardio_base.csv`: larger cardiovascular risk dataset with `cardio` target.
- `data/raw/cardiac_failure_processed.csv`: processed variant of the cardio dataset.
- `data/raw/ecg_timeseries.csv`: wide ECG timeseries data reserved for optional temporal/deep learning experiments.

The main modeling path starts with `heart_processed.csv` because its fields map directly to live risk prediction inputs. `cardio_base.csv` is used as a larger secondary dataset for comparison and robustness checks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 100)

DATA_DIR = Path('../data/raw') if Path('../data/raw').exists() else Path('data/raw')
HEART_PATH = DATA_DIR / 'heart_processed.csv'
CARDIO_PATH = DATA_DIR / 'cardio_base.csv'
CARDIO_PROCESSED_PATH = DATA_DIR / 'cardiac_failure_processed.csv'
ECG_PATH = DATA_DIR / 'ecg_timeseries.csv'

## 1. Load And Inspect Data

Missing values coded as `?` are converted to `NaN`. Numeric columns are median-imputed for EDA so plots and summary tables are stable.

In [ ]:
def load_csv(path, sep=','):
    df = pd.read_csv(path, sep=sep, na_values='?')
    unnamed_cols = [col for col in df.columns if str(col).startswith('Unnamed')]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    return df

heart = load_csv(HEART_PATH)
cardio = load_csv(CARDIO_PATH, sep=';')
cardio_processed = load_csv(CARDIO_PROCESSED_PATH)

print('heart_processed shape:', heart.shape)
print('cardio_base shape:', cardio.shape)
print('cardiac_failure_processed shape:', cardio_processed.shape)
print('ecg_timeseries exists:', ECG_PATH.exists())

display(heart.head())
display(cardio.head())

In [ ]:
def inspect_frame(df, name):
    print(f'\n{name}')
    print('-' * len(name))
    display(pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (df.isna().mean() * 100).round(2),
        'unique': df.nunique(),
    }))

inspect_frame(heart, 'heart_processed')
inspect_frame(cardio, 'cardio_base')

## 2. Clean Targets And Impute Missing Values

Targets are binary already in both selected datasets:

- `HeartDisease`: 0 = no disease, 1 = disease.
- `cardio`: 0 = no disease, 1 = disease.

If future source files contain multi-class disease labels, values greater than 0 should be converted to 1.

In [ ]:
def median_impute_numeric(df):
    cleaned = df.copy()
    for col in cleaned.columns:
        if pd.api.types.is_bool_dtype(cleaned[col]):
            cleaned[col] = cleaned[col].astype(int)
        cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')
        if cleaned[col].isna().any():
            cleaned[col] = cleaned[col].fillna(cleaned[col].median())
    return cleaned

heart_clean = median_impute_numeric(heart)
cardio_clean = median_impute_numeric(cardio)

heart_clean['target'] = (heart_clean['HeartDisease'] > 0).astype(int)
cardio_clean['target'] = (cardio_clean['cardio'] > 0).astype(int)

# Convert cardio age from days to years for interpretability.
cardio_clean['age_years'] = (cardio_clean['age'] / 365.25).round(1)

print('Heart target distribution:')
display(heart_clean['target'].value_counts().rename_axis('target').reset_index(name='count'))
print('Cardio target distribution:')
display(cardio_clean['target'].value_counts().rename_axis('target').reset_index(name='count'))

## 3. Summary Statistics

In [ ]:
display(heart_clean.drop(columns=['target']).describe().T)
display(cardio_clean[['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'target']].describe().T)

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=heart_clean, x='target', ax=axes[0])
axes[0].set_title('Heart Disease Target')
axes[0].set_xlabel('0 = No disease, 1 = Disease')

sns.countplot(data=cardio_clean, x='target', ax=axes[1])
axes[1].set_title('Cardio Target')
axes[1].set_xlabel('0 = No disease, 1 = Disease')

plt.tight_layout()
plt.show()

## 5. Correlation Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

heart_corr = heart_clean.drop(columns=['HeartDisease']).corr(numeric_only=True)
sns.heatmap(heart_corr, cmap='vlag', center=0, ax=axes[0])
axes[0].set_title('Heart Processed Correlation')

cardio_cols = ['age_years', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'target']
cardio_corr = cardio_clean[cardio_cols].corr(numeric_only=True)
sns.heatmap(cardio_corr, cmap='vlag', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Cardio Base Correlation')

plt.tight_layout()
plt.show()

## 6. Feature Distributions By Target

In [ ]:
heart_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(heart_features):
    sns.histplot(data=heart_clean, x=feature, hue='target', kde=True, bins=25, ax=axes[idx])
    axes[idx].set_title(f'{feature} by Target')

axes[-1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
cardio_features = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

sample_cardio = cardio_clean.sample(min(10000, len(cardio_clean)), random_state=42)
for idx, feature in enumerate(cardio_features):
    sns.histplot(data=sample_cardio, x=feature, hue='target', kde=True, bins=30, ax=axes[idx])
    axes[idx].set_title(f'{feature} by Target')

plt.tight_layout()
plt.show()

## 7. Age Histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=heart_clean, x='Age', hue='target', bins=25, kde=True, ax=axes[0])
axes[0].set_title('Heart Dataset Age Distribution')

sns.histplot(data=sample_cardio, x='age_years', hue='target', bins=30, kde=True, ax=axes[1])
axes[1].set_title('Cardio Dataset Age Distribution')
axes[1].set_xlabel('Age in years')

plt.tight_layout()
plt.show()

## 8. Blood Pressure And Cholesterol Boxplots

The Cleveland-style request mentioned `trestbps` and `chol`; in these provided datasets the closest equivalents are:

- `RestingBP` and `Cholesterol` in `heart_processed.csv`.
- `ap_hi`, `ap_lo`, and `cholesterol` in `cardio_base.csv`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(data=heart_clean, x='target', y='RestingBP', ax=axes[0])
axes[0].set_title('Resting BP by Heart Disease')

sns.boxplot(data=heart_clean, x='target', y='Cholesterol', ax=axes[1])
axes[1].set_title('Cholesterol by Heart Disease')

sns.boxplot(data=sample_cardio, x='target', y='ap_hi', ax=axes[2])
axes[2].set_title('Systolic BP by Cardio Target')

plt.tight_layout()
plt.show()

## 9. EDA Notes

Initial modeling recommendation:

1. Use `heart_processed.csv` as the primary demo dataset because its columns match the planned Streamlit UI.
2. Use `cardio_base.csv` for larger-sample classical model benchmarking and SHAP robustness analysis.
3. Keep `ecg_timeseries.csv` for an optional temporal experiment after the tabular pipeline is stable, because it is very wide and will need a separate modeling strategy.